# 05 Plot Results (Part 2 / Paper-Style Visualization)

Goal:

1. read only from `results/part2/` exported tables (not raw output)
2. support the expected 5 result slots: Re10k(1), DL3DV(1), 405841(A/B/C)
3. generate paper-style main figures and appendix figures
4. export publication-ready tables for report writing

Design notes:
- PSNR/SSIM: higher is better.
- LPIPS/ATE: lower is better.
- ATE magnitude is shown, but cross-dataset ATE ranking should be interpreted cautiously.
- Missing expected runs are marked as `pending` and do not break plotting.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap

CV_ROOT = Path('~/CV_Project').expanduser()
RESULTS_ROOT = CV_ROOT / 'results' / 'part2'
PLOTS_ROOT = CV_ROOT / 'plots' / 'part2'
MAIN_DIR = PLOTS_ROOT / 'main'
APP_DIR = PLOTS_ROOT / 'appendix'
TABLE_DIR = RESULTS_ROOT / 'paper_tables'

for d in [MAIN_DIR, APP_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'font.family': 'DejaVu Serif',
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 9,
    'axes.grid': True,
    'grid.color': '#d9d9d9',
    'grid.linestyle': '--',
    'grid.linewidth': 0.6,
    'axes.edgecolor': '#222222',
    'axes.linewidth': 0.8,
    'savefig.dpi': 300
})

DATASET_COLORS = {
    're10k': '#1f77b4',
    'dl3dv_2': '#ff7f0e',
    '405841': '#2ca02c',
}

DATASET_LABELS = {
    're10k': 'Re10k-1',
    'dl3dv_2': 'DL3DV-2',
    '405841': '405841',
}

print('RESULTS_ROOT =', RESULTS_ROOT)
print('PLOTS_ROOT =', PLOTS_ROOT)
print('MAIN_DIR =', MAIN_DIR)
print('APP_DIR =', APP_DIR)
print('TABLE_DIR =', TABLE_DIR)

## 1. Load csv inputs and perform strict sanity checks

In [ ]:
run_summary_csv = RESULTS_ROOT / 'final' / 'part2_run_summary.csv'
frame_metrics_csv = RESULTS_ROOT / 'frame' / 'part2_frame_metrics.csv'
dataset_summary_csv = RESULTS_ROOT / 'final' / 'part2_dataset_summary.csv'
run_inventory_csv = RESULTS_ROOT / 'qc' / 'part2_run_inventory.csv'

required_paths = [run_summary_csv, frame_metrics_csv, dataset_summary_csv, run_inventory_csv]
missing_paths = [p for p in required_paths if not p.exists()]
if missing_paths:
    msg = '\n'.join(str(p) for p in missing_paths)
    raise FileNotFoundError(
        'Missing Part2 result csv files. Run part2/notebooks/04_read_results.ipynb first. Missing:\n' + msg
    )

run_summary_df = pd.read_csv(run_summary_csv)
frame_df = pd.read_csv(frame_metrics_csv)
dataset_summary_df = pd.read_csv(dataset_summary_csv)
inventory_df = pd.read_csv(run_inventory_csv)

print('run_summary_df =', run_summary_df.shape)
print('frame_df =', frame_df.shape)
print('dataset_summary_df =', dataset_summary_df.shape)
print('inventory_df =', inventory_df.shape)

run_summary_df.head()

## 2. Build expected-5 experiment slots and availability map

In [ ]:
expected_runs = pd.DataFrame([
    {
        'display_order': 1,
        'dataset': 're10k',
        'scene_id': 'main',
        'scene_label': 'Re10k',
        'run_name_expected': 'reggs_re10k1_sr30_nv8',
        'target_sample_rate': 30,
        'target_n_views': 8,
    },
    {
        'display_order': 2,
        'dataset': 'dl3dv_2',
        'scene_id': 'main',
        'scene_label': 'DL3DV',
        'run_name_expected': 'reggs_dl3dv2_sr30_nv8',
        'target_sample_rate': 30,
        'target_n_views': 8,
    },
    {
        'display_order': 3,
        'dataset': '405841',
        'scene_id': 'A',
        'scene_label': '405841-A',
        'run_name_expected': 'reggs_405841_scene_A_sr10_nv4',
        'target_sample_rate': 10,
        'target_n_views': 4,
    },
    {
        'display_order': 4,
        'dataset': '405841',
        'scene_id': 'B',
        'scene_label': '405841-B',
        'run_name_expected': 'reggs_405841_scene_B_sr10_nv4',
        'target_sample_rate': 10,
        'target_n_views': 4,
    },
    {
        'display_order': 5,
        'dataset': '405841',
        'scene_id': 'C',
        'scene_label': '405841-C',
        'run_name_expected': 'reggs_405841_scene_C_sr10_nv4',
        'target_sample_rate': 10,
        'target_n_views': 4,
    },
])

run_summary_u = run_summary_df.drop_duplicates(subset=['dataset', 'run_name'], keep='first').copy()

plot_df = expected_runs.merge(
    run_summary_u,
    left_on=['dataset', 'run_name_expected'],
    right_on=['dataset', 'run_name'],
    how='left',
    indicator=True
)

plot_df['status'] = np.where(plot_df['_merge'].eq('both'), 'available', 'pending')
plot_df['dataset_label'] = plot_df['dataset'].map(DATASET_LABELS).fillna(plot_df['dataset'])
plot_df['slot_label'] = plot_df['scene_label']

protocol_ok = (
    (plot_df['sample_rate_from_name'] == plot_df['target_sample_rate'])
    & (plot_df['n_views_from_name'] == plot_df['target_n_views'])
)
plot_df['protocol_match'] = np.where(plot_df['status'].eq('available'), protocol_ok, np.nan)

expected_key = expected_runs[['dataset', 'run_name_expected']].rename(columns={'run_name_expected': 'run_name'})
extra_runs_df = run_summary_u.merge(expected_key, on=['dataset', 'run_name'], how='left', indicator=True)
extra_runs_df = extra_runs_df[extra_runs_df['_merge'].eq('left_only')].copy()

print('expected slots =', len(expected_runs))
print('available slots =', int((plot_df['status'] == 'available').sum()))
print('pending slots =', int((plot_df['status'] == 'pending').sum()))
print('extra runs not in expected-5 =', len(extra_runs_df))

plot_df[['display_order', 'dataset', 'scene_label', 'run_name_expected', 'status', 'protocol_match']]

## 3. Export paper tables

In [ ]:
table_cols = [
    'display_order', 'dataset_label', 'scene_label', 'run_name_expected', 'status',
    'target_sample_rate', 'target_n_views',
    'n_train_frames', 'n_test_frames',
    'train_avg_psnr', 'test_avg_psnr', 'psnr_drop_train_to_test',
    'train_avg_ssim', 'test_avg_ssim', 'ssim_drop_train_to_test',
    'train_avg_lpips', 'test_avg_lpips', 'lpips_increase_train_to_test',
    'ate_rmse', 'ate_mean', 'ate_median', 'ate_std',
    'generalization_risk', 'protocol_match'
]
for c in table_cols:
    if c not in plot_df.columns:
        plot_df[c] = np.nan

paper_table_expected5 = plot_df[table_cols].sort_values('display_order').copy()
paper_table_available = paper_table_expected5[paper_table_expected5['status'].eq('available')].copy()

out_expected5 = TABLE_DIR / 'part2_expected5_main_table.csv'
out_available = TABLE_DIR / 'part2_available_only_main_table.csv'
out_extras = TABLE_DIR / 'part2_extra_runs_not_in_expected5.csv'

paper_table_expected5.to_csv(out_expected5, index=False)
paper_table_available.to_csv(out_available, index=False)
extra_runs_df.to_csv(out_extras, index=False)

print('out_expected5 =', out_expected5)
print('out_available =', out_available)
print('out_extras =', out_extras)

paper_table_expected5

## 4. Main figure A: Test metrics + ATE (expected-5 slots with pending marks)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14.5, 8.2), sharex=True)
axes = axes.flatten()

metric_specs = [
    ('test_avg_psnr', 'Test PSNR (higher is better)'),
    ('test_avg_ssim', 'Test SSIM (higher is better)'),
    ('test_avg_lpips', 'Test LPIPS (lower is better)'),
    ('ate_rmse', 'ATE RMSE (lower is better, log-scale)'),
]

plot_df_sorted = plot_df.sort_values('display_order').reset_index(drop=True)
x = np.arange(len(plot_df_sorted))

for ax, (metric_col, metric_title) in zip(axes, metric_specs):
    for i, row in plot_df_sorted.iterrows():
        dataset = row['dataset']
        color = DATASET_COLORS.get(dataset, '#7f7f7f')
        if row['status'] == 'available' and pd.notna(row.get(metric_col, np.nan)):
            ax.scatter(i, row[metric_col], s=72, color=color, edgecolor='black', linewidth=0.45, zorder=3)
        else:
            ax.axvspan(i - 0.40, i + 0.40, color='#f2f2f2', alpha=0.8, zorder=0)
            ax.text(i, 0.04, 'pending', rotation=90, va='bottom', ha='center',
                    transform=ax.get_xaxis_transform(), fontsize=8, color='#7f7f7f')

    ax.set_title(metric_title)
    ax.grid(True, axis='y')

    if metric_col == 'ate_rmse':
        positive = plot_df_sorted.loc[
            (plot_df_sorted['status'] == 'available') & (plot_df_sorted['ate_rmse'] > 0),
            'ate_rmse'
        ]
        if len(positive) > 0:
            ax.set_yscale('log')

for ax in axes:
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df_sorted['slot_label'], rotation=20, ha='right')

dataset_legend = [
    Line2D([0], [0], marker='o', linestyle='None', markersize=7,
           markerfacecolor=DATASET_COLORS[k], markeredgecolor='black', label=DATASET_LABELS.get(k, k))
    for k in ['re10k', 'dl3dv_2', '405841']
]
pending_legend = [Patch(facecolor='#f2f2f2', edgecolor='none', label='Pending slot')]

fig.legend(handles=dataset_legend + pending_legend, loc='upper center', ncol=4, frameon=False)
fig.suptitle('Part2 Main Metrics by Expected Slots', y=0.99, fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.95))

out_main_a_png = MAIN_DIR / 'part2_main_test_metrics_and_ate.png'
out_main_a_pdf = MAIN_DIR / 'part2_main_test_metrics_and_ate.pdf'
fig.savefig(out_main_a_png, bbox_inches='tight')
fig.savefig(out_main_a_pdf, bbox_inches='tight')
plt.show()
plt.close(fig)

print('saved:', out_main_a_png)
print('saved:', out_main_a_pdf)

## 5. Main figure B: Train-Test gap dumbbell plots

In [ ]:
plot_df_sorted = plot_df.sort_values('display_order').reset_index(drop=True)
y_pos = np.arange(len(plot_df_sorted))[::-1]

gap_specs = [
    ('train_avg_psnr', 'test_avg_psnr', 'PSNR: train vs test'),
    ('train_avg_ssim', 'test_avg_ssim', 'SSIM: train vs test'),
    ('train_avg_lpips', 'test_avg_lpips', 'LPIPS: train vs test'),
]

fig, axes = plt.subplots(1, 3, figsize=(17, 6), sharey=True)

for ax, (train_col, test_col, title) in zip(axes, gap_specs):
    for i, row in plot_df_sorted.iterrows():
        y = y_pos[i]
        dataset = row['dataset']
        color = DATASET_COLORS.get(dataset, '#7f7f7f')

        if row['status'] == 'available' and pd.notna(row.get(train_col, np.nan)) and pd.notna(row.get(test_col, np.nan)):
            train_v = row[train_col]
            test_v = row[test_col]
            ax.plot([train_v, test_v], [y, y], color=color, linewidth=1.4, alpha=0.9, zorder=2)
            ax.scatter(train_v, y, s=52, facecolors='white', edgecolors=color, linewidth=1.4, zorder=3)
            ax.scatter(test_v, y, s=52, facecolors=color, edgecolors='black', linewidth=0.45, zorder=3)
        else:
            ax.text(0.02, y, 'pending', transform=ax.get_yaxis_transform(),
                    va='center', ha='left', fontsize=8, color='#8a8a8a')

    ax.set_title(title)
    ax.grid(True, axis='x')

axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(plot_df_sorted['slot_label'])
for ax in axes[1:]:
    ax.set_yticks(y_pos)
    ax.set_yticklabels([])

legend_handles = [
    Line2D([0], [0], marker='o', color='none', markerfacecolor='white', markeredgecolor='black', markersize=7, label='Train avg'),
    Line2D([0], [0], marker='o', color='none', markerfacecolor='black', markeredgecolor='black', markersize=7, label='Test avg'),
]
fig.legend(handles=legend_handles, loc='upper center', ncol=2, frameon=False)
fig.suptitle('Part2 Generalization Gap Diagnostics (Train vs Test)', y=0.98, fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.93))

out_main_b_png = MAIN_DIR / 'part2_main_train_test_gap_dumbbell.png'
out_main_b_pdf = MAIN_DIR / 'part2_main_train_test_gap_dumbbell.pdf'
fig.savefig(out_main_b_png, bbox_inches='tight')
fig.savefig(out_main_b_pdf, bbox_inches='tight')
plt.show()
plt.close(fig)

print('saved:', out_main_b_png)
print('saved:', out_main_b_pdf)

## 6. Main figure C: quality-geometry relationship (ATE vs Test PSNR)

In [ ]:
scatter_df = plot_df[(plot_df['status'] == 'available')].copy()
scatter_df = scatter_df[scatter_df['ate_rmse'].notna() & scatter_df['test_avg_psnr'].notna()]
scatter_df = scatter_df[scatter_df['ate_rmse'] > 0]

fig, ax = plt.subplots(figsize=(8.2, 6.2))

for _, row in scatter_df.iterrows():
    dataset = row['dataset']
    color = DATASET_COLORS.get(dataset, '#7f7f7f')
    ax.scatter(row['ate_rmse'], row['test_avg_psnr'], s=86, color=color, edgecolor='black', linewidth=0.45, zorder=3)

    label = row['scene_label']
    if pd.notna(row.get('n_test_frames', np.nan)):
        label = f"{label} (n={int(row['n_test_frames'])})"
    ax.annotate(label, (row['ate_rmse'], row['test_avg_psnr']),
                xytext=(5, 6), textcoords='offset points', fontsize=9)

if len(scatter_df) > 0:
    ax.set_xscale('log')

ax.set_xlabel('ATE RMSE (log scale, lower is better)')
ax.set_ylabel('Test PSNR (higher is better)')
ax.set_title('Quality vs Geometry: Part2 Runs')
ax.grid(True, which='both', axis='both')

text_note = 'Caution: absolute ATE magnitudes may not be directly comparable across datasets.'
ax.text(0.02, -0.18, text_note, transform=ax.transAxes, fontsize=9, color='#555555')

out_main_c_png = MAIN_DIR / 'part2_main_quality_vs_geometry_scatter.png'
out_main_c_pdf = MAIN_DIR / 'part2_main_quality_vs_geometry_scatter.pdf'
fig.savefig(out_main_c_png, bbox_inches='tight')
fig.savefig(out_main_c_pdf, bbox_inches='tight')
plt.show()
plt.close(fig)

print('saved:', out_main_c_png)
print('saved:', out_main_c_pdf)

## 7. Appendix figure: test-frame metric distributions

In [ ]:
available_slots = plot_df[plot_df['status'] == 'available'][['dataset', 'run_name_expected', 'scene_label', 'display_order']].copy()
available_slots = available_slots.rename(columns={'run_name_expected': 'run_name'})

frame_plot_df = frame_df.merge(available_slots, on=['dataset', 'run_name'], how='inner')
frame_plot_df = frame_plot_df[frame_plot_df['split'] == 'test'].copy()

metric_cols = [
    ('psnr', 'Test frame PSNR'),
    ('ssim', 'Test frame SSIM'),
    ('lpips', 'Test frame LPIPS'),
]

fig, axes = plt.subplots(3, 1, figsize=(14, 10.5), sharex=True)

ordered_scenes = available_slots.sort_values('display_order')['scene_label'].tolist()
scene_to_dataset = {
    r.scene_label: r.dataset
    for r in available_slots.drop_duplicates('scene_label').itertuples()
}

for ax, (metric_col, title) in zip(axes, metric_cols):
    data_arrays = []
    box_colors = []
    for scene in ordered_scenes:
        arr = frame_plot_df.loc[frame_plot_df['scene_label'] == scene, metric_col].dropna().values
        data_arrays.append(arr)
        ds = scene_to_dataset.get(scene, 'unknown')
        box_colors.append(DATASET_COLORS.get(ds, '#7f7f7f'))

    if len(data_arrays) > 0:
        bp = ax.boxplot(data_arrays, patch_artist=True, widths=0.58, showfliers=False)
        for patch, color in zip(bp['boxes'], box_colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.23)
            patch.set_edgecolor(color)
            patch.set_linewidth(1.0)

        for i, arr in enumerate(data_arrays, start=1):
            if len(arr) == 0:
                continue
            jitter = np.random.normal(loc=0.0, scale=0.04, size=len(arr))
            color = box_colors[i - 1]
            ax.scatter(np.full(len(arr), i) + jitter, arr, s=16, color=color, alpha=0.62, zorder=3, linewidth=0)

    ax.set_title(title)
    ax.grid(True, axis='y')

axes[-1].set_xticks(np.arange(1, len(ordered_scenes) + 1))
axes[-1].set_xticklabels(ordered_scenes, rotation=20, ha='right')

fig.suptitle('Frame-Level Distribution (Test split only)', y=0.99, fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.96))

out_app_a_png = APP_DIR / 'part2_appendix_test_frame_distributions.png'
out_app_a_pdf = APP_DIR / 'part2_appendix_test_frame_distributions.pdf'
fig.savefig(out_app_a_png, bbox_inches='tight')
fig.savefig(out_app_a_pdf, bbox_inches='tight')
plt.show()
plt.close(fig)

print('saved:', out_app_a_png)
print('saved:', out_app_a_pdf)

## 8. Appendix figure: expected-5 completion map

In [ ]:
status_map_df = plot_df.sort_values('display_order').copy()
status_vals = status_map_df['status'].map({'pending': 0, 'available': 1}).to_numpy().reshape(1, -1)

fig, ax = plt.subplots(figsize=(12, 2.8))
cmap = ListedColormap(['#eeeeee', '#4C9F70'])
ax.imshow(status_vals, cmap=cmap, vmin=0, vmax=1, aspect='auto')

ax.set_xticks(np.arange(status_map_df.shape[0]))
ax.set_xticklabels(status_map_df['scene_label'], rotation=20, ha='right')
ax.set_yticks([0])
ax.set_yticklabels(['Status'])

for i, row in status_map_df.iterrows():
    txt = 'OK' if row['status'] == 'available' else 'pending'
    ax.text(i, 0, txt, ha='center', va='center', fontsize=9,
            color='white' if txt == 'OK' else '#666666', fontweight='bold' if txt == 'OK' else 'normal')

for x_sep in [0.5, 1.5]:
    ax.axvline(x=x_sep, color='white', linewidth=2)

ax.set_title('Expected-5 Experiment Coverage')
fig.tight_layout()

out_app_b_png = APP_DIR / 'part2_appendix_expected5_coverage_matrix.png'
out_app_b_pdf = APP_DIR / 'part2_appendix_expected5_coverage_matrix.pdf'
fig.savefig(out_app_b_png, bbox_inches='tight')
fig.savefig(out_app_b_pdf, bbox_inches='tight')
plt.show()
plt.close(fig)

print('saved:', out_app_b_png)
print('saved:', out_app_b_pdf)

## 9. Show generated files

In [ ]:
def show_files(path: Path):
    print('\n==', path, '==')
    if not path.exists():
        print('missing')
        return
    files = sorted([p for p in path.iterdir() if p.is_file()])
    if not files:
        print('(empty)')
        return
    for f in files:
        print('-', f.name)

show_files(MAIN_DIR)
show_files(APP_DIR)
show_files(TABLE_DIR)